# NDR — Notebook 10: TRACE Consistency — VERIFY trước, train sau

## Mục tiêu bước này: KIỂM CHỨNG giả thuyết, KHÔNG train vội
Theo TRACE (arXiv 2503.15293, verify từ paper): box POISON có confidence ỔN ĐỊNH bất thường khi đổi background; box streak THẬT thì confidence DAO ĐỘNG theo context. TRACE tính CTC per-object, +30% AUROC.

**Nhưng giả thuyết đó đúng với poison CỦA TA hay không thì CHƯA biết** (data khác paper). NB này đo trực tiếp trên 20 ảnh unlearn (đã biết box nào là poison) để trả lời DỨT KHOÁT:

**Câu hỏi kiểm chứng:** consistency-score có TÁCH được box-poison khỏi box-streak-thật không?
- Nếu CÓ (AUROC cao) → cách B khả thi: dùng consistency làm clean-ref để sửa weight. Build NB10-train.
- Nếu KHÔNG → dừng, không đổ công train. Tiết kiệm như bài học NB7/NB9.

Đây đúng tinh thần experiment-agent: verify hypothesis TRƯỚC khi build.

## 1. Setup

In [1]:
!pip install -q 'setuptools<81'
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'
!pip install -q ensemble_boxes

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 6.6 MB/s eta 0:00:00


In [2]:
import copy, json, itertools, pickle, random, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
import cv2, numpy as np, pandas as pd, torch, torch.nn as nn
from scipy.optimize import linear_sum_assignment
from tqdm import tqdm
from ensemble_boxes import weighted_boxes_fusion
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.data import (DatasetCatalog, DatasetMapper, MetadataCatalog,
                             build_detection_train_loader, detection_utils as utils)
from detectron2.structures import BoxMode
from detectron2.engine import DefaultPredictor, DefaultTrainer
from detectron2.modeling import build_model
import logging; logging.getLogger('detectron2').setLevel(logging.ERROR)

ROOT = '/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models'
POISONED_WEIGHTS = f'{ROOT}/poisoned_model/poisoned_model.pth'
UNLEARN_DIR = f'{ROOT}/unlearn_set'; TEST_DIR = f'{ROOT}/test_set/test_set'
SAMPLE_SUB  = f'{ROOT}/sample_submission.csv'
WORK = Path('/kaggle/working'); WORK.mkdir(exist_ok=True)
SUBMISSION_PATH = str(WORK / 'submission.csv')

BASE_CONFIG          = 'COCO-Detection/retinanet_R_50_FPN_3x.yaml'
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES         = [[16], [32], [64], [128], [256]]
NUM_CLASSES = 1; IMG_W = IMG_H = 1024; BATCH_SIZE = 4

A_FACTOR = 10.0
IOU_THRESHOLDS = np.round(np.arange(0.2, 0.91, 0.1), 2)
CAND_THRESH = 0.05
SUBMIT_THRESH = 0.2
CLEAN_CONF_FLOOR = 0.2
MATCH_IOU = 0.3
POISON_OVERLAP_IOU = 0.1   # detection trùng poison nếu IoU>=0.1 (loại khỏi retain)
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Setup OK | device:', DEVICE)

Setup OK | device: cuda


## 2. Metric core

In [3]:
def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0: return np.zeros((len(a), len(b)), np.float32)
    ix1 = np.maximum(a[:,0:1], b[None,:,0]); iy1 = np.maximum(a[:,1:2], b[None,:,1])
    ix2 = np.minimum(a[:,2:3], b[None,:,2]); iy2 = np.minimum(a[:,3:4], b[None,:,3])
    iw = np.clip(ix2-ix1,0,None); ih = np.clip(iy2-iy1,0,None); inter = iw*ih
    aa = ((a[:,2]-a[:,0])*(a[:,3]-a[:,1]))[:,None]; ab = ((b[:,2]-b[:,0])*(b[:,3]-b[:,1]))[None,:]
    return np.where(aa+ab-inter>0, inter/(aa+ab-inter+1e-9), 0.0).astype(np.float32)

def acadd_t(cb, cs, pb, ps, t, cip):
    # SPEC BTC: mỗi clean box khớp de-pois box IoU CAO NHẤT (>=t); các de-pois box thừa = FP.
    ious = iou_matrix(cb, pb); cost = 0.0; mc=set(); mp=set()
    if ious.size:
        pairs = sorted(((ious[i,j], i, j) for i in range(len(cb)) for j in range(len(pb)) if ious[i,j] >= t), reverse=True)
        for _, i, j in pairs:
            if i in mc or j in mp: continue
            mc.add(i); mp.add(j); d = float(cs[i]) - float(ps[j])
            if cip[i]: cost += (d/A_FACTOR) if d>0 else -d
            else:      cost += d if d>0 else (-d/A_FACTOR)
    cost += sum(float(cs[i]) for i in range(len(cs)) if i not in mc)   # FN: clean chưa khớp
    cost += sum(float(ps[j]) for j in range(len(ps)) if j not in mp)   # FP: de-pois box thừa
    return cost

def macadd(clean_ref, pred, poison_flags=None):
    w = IOU_THRESHOLDS/IOU_THRESHOLDS.sum(); per=[]
    for key,(cb,cs) in clean_ref.items():
        keep = cs>CLEAN_CONF_FLOOR; cb2,cs2 = cb[keep], cs[keep]
        cip = (poison_flags.get(key, np.zeros(len(cb),bool))[keep] if poison_flags else np.zeros(len(cs2),bool))
        pb,ps = pred.get(key,(np.zeros((0,4),np.float32),np.zeros(0,np.float32)))
        per.append(sum(wi*acadd_t(cb2,cs2,pb,ps,t,cip) for wi,t in zip(w,IOU_THRESHOLDS)))
    return float(np.mean(per)) if per else 0.0

bx = lambda x,y,w,h: np.array([[x,y,x+w,y+h]],np.float32); E=(np.zeros((0,4),np.float32),np.zeros(0,np.float32))
cl = {'i':(bx(10,10,40,40), np.array([0.6],np.float32))}
assert abs(macadd(cl,{'i':E})-0.6)<1e-6                                   # miss streak = FN full
assert abs(macadd(cl,{'i':(bx(10,10,40,40),np.array([0.2]))})-0.4)<1e-6   # undershoot conf = full
assert abs(macadd({'i':E},{'i':(bx(5,5,20,20),np.array([0.5]))})-0.5)<1e-6 # fire on nothing = FP full
print('maCADD unit tests passed')

maCADD unit tests passed


## 3. Load data + poisoned model

In [4]:
def load_image(path):
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im.dtype==np.uint16: im = im.astype(np.float32)/65535.0
    im = np.clip(im*255,0,255).astype(np.float32)
    if im.ndim==2: im = np.repeat(im[:,:,None],3,axis=2)
    return im
def build_cfg(weights, score_thresh=CAND_THRESH, output_dir=None):
    cfg = get_cfg(); cfg.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))
    cfg.MODEL.WEIGHTS = weights; cfg.MODEL.RETINANET.NUM_CLASSES = NUM_CLASSES
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]; cfg.MODEL.ANCHOR_GENERATOR.SIZES = ANCHOR_SIZES
    cfg.MODEL.RETINANET.SCORE_THRESH_TEST = score_thresh; cfg.MODEL.DEVICE = DEVICE; cfg.INPUT.FORMAT = 'RGB'
    if output_dir: cfg.OUTPUT_DIR = str(output_dir); Path(output_dir).mkdir(parents=True, exist_ok=True)
    return cfg
def build_predictor(weights, score_thresh=CAND_THRESH): return DefaultPredictor(build_cfg(weights, score_thresh))
def predict_image(predictor, img):
    inst = predictor(img)['instances'].to('cpu')
    return inst.pred_boxes.tensor.numpy().astype(np.float32), inst.scores.numpy().astype(np.float32)

with open(Path(UNLEARN_DIR)/'annotations_coco.json') as f: coco = json.load(f)
stem_by_id = {im['id']: Path(im['file_name']).stem for im in coco['images']}
poison_boxes_by_stem = {}
for a in coco['annotations']:
    x,y,w,h = a['bbox']; poison_boxes_by_stem.setdefault(stem_by_id[a['image_id']], []).append([x,y,x+w,y+h])
poison_boxes_by_stem = {k: np.array(v,np.float32) for k,v in poison_boxes_by_stem.items()}
poison_xywh = {}
for a in coco['annotations']: poison_xywh.setdefault(a['image_id'], []).append(a['bbox'])
unlearn_files = sorted(Path(UNLEARN_DIR).glob('*.png'), key=lambda p:int(p.stem))
unlearn_imgs = {p.stem: load_image(p) for p in unlearn_files}
test_files = sorted(Path(TEST_DIR).glob('*.png'), key=lambda p:int(p.stem))
poisoned_predictor = build_predictor(POISONED_WEIGHTS, CAND_THRESH)
print('unlearn:', len(unlearn_files), '| test:', len(test_files))

Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


unlearn: 20 | test: 2000


## 4. Consistency score per-box (TRACE-style, black-box)
Với mỗi box detect: đo confidence của nó qua K biến đổi BACKGROUND (giữ nguyên vùng box, thay đổi ngoài box). Poison → confidence ổn định (var thấp). Streak thật → dao động (var cao). Trả về var confidence mỗi box.

In [5]:
def _bg_transform(img, box, mode, rng):
    '''Giữ nguyên vùng trong box, biến đổi NGOÀI box (background).'''
    x1,y1,x2,y2 = [int(v) for v in box]
    out = img.copy(); inside = out[y1:y2, x1:x2].copy()
    if mode=='noise':
        out = np.clip(out + rng.normal(0,12,out.shape).astype(np.float32),0,255)
    elif mode=='bright':
        out = np.clip(out*rng.uniform(0.6,1.4),0,255)
    elif mode=='blur':
        out = cv2.GaussianBlur(out,(0,0),rng.uniform(1.0,3.0))
    elif mode=='shuffle':
        h,w = out.shape[:2]; gh,gw = h//4, w//4
        tiles=[out[i*gh:(i+1)*gh, j*gw:(j+1)*gw].copy() for i in range(4) for j in range(4)]
        order=list(range(16)); rng.shuffle(order); k=0
        for i in range(4):
            for j in range(4):
                t=tiles[order[k]]; k+=1
                if t.shape==out[i*gh:(i+1)*gh, j*gw:(j+1)*gw].shape: out[i*gh:(i+1)*gh, j*gw:(j+1)*gw]=t
    if y2>y1 and x2>x1: out[y1:y2, x1:x2] = inside   # phục hồi vùng box (giữ foreground)
    return out

def box_consistency(img, box, base_conf, K=6, seed=0):
    '''Var của confidence box qua K biến đổi background. Thấp = ổn định = nghi poison.'''
    rng=np.random.default_rng(seed); confs=[base_conf]
    modes=['noise','bright','blur','shuffle','noise','bright']
    for t in range(K):
        im2=_bg_transform(img, box, modes[t%len(modes)], rng)
        b2,s2=predict_image(poisoned_predictor, im2)
        if len(b2):
            ious=iou_matrix(np.array([box],np.float32), b2)[0]; j=ious.argmax()
            confs.append(float(s2[j]) if ious[j]>=0.3 else 0.0)
        else: confs.append(0.0)
    return float(np.var(confs)), float(np.mean(confs))
print('box_consistency defined.')

box_consistency defined.


## 5. KIỂM CHỨNG trên 20 ảnh unlearn (ground-truth poison đã biết)
Chia box detect thành 2 nhóm: TRÙNG poison-GT (poison) vs KHÔNG trùng (nghi streak thật). So var-consistency 2 nhóm + tính AUROC. AUROC cao = tách được = cách B khả thi.

In [6]:
from sklearn.metrics import roc_auc_score
poison_vars=[]; clean_vars=[]
for stem in tqdm(list(unlearn_imgs), desc='consistency'):
    img=unlearn_imgs[stem]; b,s=predict_image(poisoned_predictor, img)
    if not len(b): continue
    pb=poison_boxes_by_stem.get(stem, np.zeros((0,4),np.float32))
    for k in range(len(b)):
        var,mean=box_consistency(img, b[k], float(s[k]), K=6, seed=k)
        is_pois = len(pb) and iou_matrix(b[k:k+1], pb).max()>=0.1
        (poison_vars if is_pois else clean_vars).append(var)
pv=np.array(poison_vars); cv=np.array(clean_vars)
print(f'poison boxes={len(pv)} var mean={pv.mean():.5f} | clean boxes={len(cv)} var mean={cv.mean():.5f}')
if len(pv)>1 and len(cv)>1:
    y=np.r_[np.ones(len(pv)), np.zeros(len(cv))]
    # nghi poison = var THẤP -> score = -var để AUROC theo 'poison positive'
    score=np.r_[-pv, -cv]
    auc=roc_auc_score(y, score)
    print(f'AUROC (tách poison vs clean bằng consistency) = {auc:.3f}')
    print('-> >0.75: cách B KHẢ THI (consistency tách được). ~0.5: KHÔNG tách được, dừng.')
else:
    print('Không đủ box 2 nhóm để tính AUROC.')

consistency: 100%|██████████| 20/20 [01:00<00:00,  3.04s/it]

poison boxes=52 var mean=0.02751 | clean boxes=38 var mean=0.01909
AUROC (tách poison vs clean bằng consistency) = 0.441
-> >0.75: cách B KHẢ THI (consistency tách được). ~0.5: KHÔNG tách được, dừng.


## 6. Kết luận kiểm chứng
Đọc AUROC ở cell trên:
- **AUROC ≥ 0.75:** consistency tách được poison khỏi streak thật → giả thuyết đúng với data của ta → build NB10-train (dùng consistency đánh dấu box poison, tạo clean-ref, sửa weight). ĐÂY là hướng out-of-box + co-author.
- **AUROC ~0.5-0.6:** KHÔNG tách được → poison của ta không có đặc tính consistency như TRACE giả định → DỪNG, quay lại NB4-v2 (hướng chắc ăn). Không đổ công train.

Bước verify này rẻ (~vài phút, chỉ inference), tránh lặp sai lầm NB7/NB9 (build train rồi mới biết fail).

In [7]:
print('='*60)
print('KIỂM CHỨNG TRACE-consistency cho data NDR:')
if len(pv)>1 and len(cv)>1:
    print(f'  poison var={pv.mean():.5f} | clean var={cv.mean():.5f} | AUROC={auc:.3f}')
    if auc>=0.75: print('  -> KHẢ THI. Xây NB10-train (consistency -> clean-ref -> sửa weight).')
    elif auc>=0.62: print('  -> BÁN PHẦN. Có tín hiệu nhưng yếu; cân nhắc kết hợp với activation-cue.')
    else: print('  -> KHÔNG tách được. Dừng hướng TRACE, quay lại NB4-v2.')
else: print('  thiếu dữ liệu.')

KIỂM CHỨNG TRACE-consistency cho data NDR:
  poison var=0.02751 | clean var=0.01909 | AUROC=0.441
  -> KHÔNG tách được. Dừng hướng TRACE, quay lại NB4-v2.
